# Single-Node Inference

Establish a controlled denominator for all later distributed comparisons.

## Objectives

- Launch or connect to one server and verify model identity and readiness.
- Measure startup separately from steady-state latency and throughput.
- Sweep controlled prompt and output lengths while recording memory and correctness.
- Retain raw per-request results, including failures, before aggregation.

## Background

A single-node baseline defines the denominator needed to interpret distributed results. Startup, warm-up, and steady state require separate measurement boundaries.

## Prediction

For one request at a time on a single DGX Spark, inference behavior should differ between prompt processing and autoregressive generation.

Specifically:

- increasing prompt length while keeping generated length fixed should primarily increase time to first token;
- increasing generated length while keeping prompt length fixed should primarily increase total latency;
- output-token throughput should be relatively stable across sufficiently long generations, but short generations should show lower apparent throughput because fixed request and scheduling overheads make up a larger fraction of total latency;
- the first successful request after server readiness should be slower than later steady-state requests because runtime initialization, kernel loading, graph construction, cache population, or similar one-time work may still occur;
- repeated deterministic requests with the same prompt and sampling configuration should return the same token sequence unless the serving stack introduces nondeterminism;
- no request in the initial single-request benchmark should fail or exceed the configured timeout.

These predictions are falsified if prompt length has no measurable relationship with time to first token, generated length has no measurable relationship with total latency, warm-up requests are not distinguishable from measured requests, or nominally deterministic repeated requests return inconsistent token sequences.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference


## Experiment

Complete configuration placeholders before running active measurement cells.

### Benchmark configuration and deterministic prompts

In [1]:
from dataclasses import asdict, dataclass
from itertools import product

import pandas as pd


MODEL_ID = "google/gemma-4-E4B-it"
MODEL_REVISION = "ee0ef6023621cff504d758262d4e04895a5af4a2"

HOST = "127.0.0.1"
PORT = 8000
ENDPOINT = f"http://{HOST}:{PORT}"
MODEL = MODEL_ID

# Start with an externally managed server. This keeps server launch and
# inference measurement separate during the first experiment.
SERVER_MODE = "external"

# Initial factorial workload:
#
# - 32 tokens represents a small interactive prompt.
# - 512 tokens exposes a meaningful prefill difference without making the
#   first experiment unnecessarily expensive.
# - 16 generated tokens emphasizes fixed request overhead.
# - 128 generated tokens provides a more useful decode-throughput interval.
PROMPT_TOKEN_COUNTS = (32, 512)
GENERATED_TOKEN_COUNTS = (16, 128)

WARMUP_COUNT = 3
REPETITIONS = 5
REQUEST_TIMEOUT_S = 300.0

SAMPLING = {
    "temperature": 0.0,
    "top_p": 1.0,
    "seed": 20260806,
}


@dataclass(frozen=True, slots=True)
class BenchmarkConfiguration:
    endpoint: str
    model: str
    model_revision: str
    server_mode: str
    prompt_token_counts: tuple[int, ...]
    generated_token_counts: tuple[int, ...]
    warmup_count: int
    repetitions: int
    request_timeout_s: float
    temperature: float
    top_p: float
    seed: int


benchmark_configuration = BenchmarkConfiguration(
    endpoint=ENDPOINT,
    model=MODEL,
    model_revision=MODEL_REVISION,
    server_mode=SERVER_MODE,
    prompt_token_counts=PROMPT_TOKEN_COUNTS,
    generated_token_counts=GENERATED_TOKEN_COUNTS,
    warmup_count=WARMUP_COUNT,
    repetitions=REPETITIONS,
    request_timeout_s=REQUEST_TIMEOUT_S,
    temperature=SAMPLING["temperature"],
    top_p=SAMPLING["top_p"],
    seed=SAMPLING["seed"],
)

pd.Series(asdict(benchmark_configuration), name="value")

endpoint                                     http://127.0.0.1:8000
model                                        google/gemma-4-E4B-it
model_revision            ee0ef6023621cff504d758262d4e04895a5af4a2
server_mode                                               external
prompt_token_counts                                      (32, 512)
generated_token_counts                                   (16, 128)
warmup_count                                                     3
repetitions                                                      5
request_timeout_s                                            300.0
temperature                                                    0.0
top_p                                                          1.0
seed                                                      20260806
Name: value, dtype: object

### Measurement contract

This notebook uses one request at a time. It does not measure batching, concurrency, queueing, or distributed execution.

For each request:

- **Prompt tokens** are the input token count reported by the server, checked against the intended workload size.
- **Output tokens** are the generated token count reported by the server.
- **Time to first token (TTFT)** is the elapsed monotonic wall-clock time from immediately before request submission until the first non-empty streamed token or text fragment is received.
- **End-to-end latency** is the elapsed monotonic wall-clock time from immediately before request submission until the complete response stream has been consumed.
- **Generation interval** is `end-to-end latency - TTFT`.
- **Output-token throughput** is the number of returned output tokens divided by the generation interval.
- **End-to-end token throughput** is the number of returned output tokens divided by end-to-end latency.

Warm-up requests are retained separately and are never included in steady-state aggregates.

All raw request records, including failed requests and token-count mismatches, are retained before aggregation.

In [ ]:
def validate_benchmark_configuration(
    configuration: BenchmarkConfiguration,
) -> None:
    if not configuration.endpoint.startswith(("http://", "https://")):
        raise ValueError("endpoint must be an HTTP or HTTPS URL")

    if not configuration.model:
        raise ValueError("model must not be empty")

    if configuration.server_mode not in {"external", "notebook_subprocess"}:
        raise ValueError("server_mode must be 'external' or 'notebook_subprocess'")

    for name, values in (
        ("prompt_token_counts", configuration.prompt_token_counts),
        ("generated_token_counts", configuration.generated_token_counts),
    ):
        if not values:
            raise ValueError(f"{name} must not be empty")
        if any(
            isinstance(value, bool) or not isinstance(value, int) or value <= 0
            for value in values
        ):
            raise ValueError(f"{name} must contain positive integers")
        if len(values) != len(set(values)):
            raise ValueError(f"{name} must not contain duplicates")

    if configuration.warmup_count < 0:
        raise ValueError("warmup_count must be non-negative")

    if configuration.repetitions <= 0:
        raise ValueError("repetitions must be positive")

    if configuration.request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be positive")

    if configuration.temperature < 0:
        raise ValueError("temperature must be non-negative")

    if not 0 < configuration.top_p <= 1:
        raise ValueError("top_p must be in the interval (0, 1]")


validate_benchmark_configuration(benchmark_configuration)

workload_matrix = pd.DataFrame(
    [
        {
            "prompt_tokens_target": prompt_tokens,
            "requested_output_tokens": generated_tokens,
        }
        for prompt_tokens, generated_tokens in product(
            PROMPT_TOKEN_COUNTS,
            GENERATED_TOKEN_COUNTS,
        )
    ]
)

expected_measured_requests = len(workload_matrix) * REPETITIONS

print(f"Warm-up requests: {WARMUP_COUNT}")
print(f"Workload configurations: {len(workload_matrix)}")
print(f"Repetitions per configuration: {REPETITIONS}")
print(f"Expected measured requests: {expected_measured_requests}")

workload_matrix

Warm-up requests: 3
Workload configurations: 4
Repetitions per configuration: 5
Expected measured requests: 20


,prompt_tokens_target,requested_output_tokens
0,32,16
1,32,128
2,512,16
3,512,128


### Server lifecycle

For notebook-managed mode, construct an argument list only after selecting a model. Retain the exact `Popen` handle and terminate only that process during cleanup.

In [ ]:
SERVER_ARGUMENTS = None  # Example shape: [executable, *validated_arguments]
server_process = None


def build_server_arguments(*, executable: str, model: str, port: int) -> list[str]:
    if not executable or not model or not 1 <= port <= 65535:
        raise ValueError("Executable, model, and a valid port are required")
    return [executable, "serve", model, "--port", str(port)]

### Readiness and request scaffolds

These functions remain inactive until endpoint and model configuration are filled in.

In [ ]:
import time
from typing import Any


def wait_until_ready(endpoint: str, timeout_s: float) -> dict[str, Any]:
    """TODO: Poll a validated readiness URL with bounded retries."""
    raise NotImplementedError


def send_completion_request(
    endpoint: str, model: str, prompt: str, max_tokens: int
) -> dict[str, Any]:
    """TODO: POST an OpenAI-compatible request and retain timing, usage, and errors."""
    raise NotImplementedError

### Warm-up and raw result schema

In [ ]:
WARMUP_RESULTS = []  # Run separately; never mix with measured trials.

raw_result_columns = (
    "request_id",
    "trial",
    "prompt_id",
    "prompt_tokens",
    "requested_output_tokens",
    "returned_output_tokens",
    "started_monotonic_s",
    "first_token_monotonic_s",
    "completed_monotonic_s",
    "ttft_s",
    "latency_s",
    "memory_bytes",
    "output_text",
    "output_correct",
    "status",
    "error",
)
raw_results = pd.DataFrame(columns=raw_result_columns)
raw_results

### Aggregation schema

In [ ]:
aggregate_columns = (
    "configuration",
    "prompt_tokens",
    "requested_output_tokens",
    "metric",
    "minimum",
    "median",
    "p90",
    "p99",
    "count",
    "failures",
)
aggregates = pd.DataFrame(columns=aggregate_columns)
aggregates

### Cleanup guidance

Stop only the `server_process` handle created by this notebook, first requesting graceful termination and then applying a bounded wait. Never use unscoped process-kill commands.

## Observations

TODO: Record only facts produced by the saved outputs of this notebook.

## Explanation

TODO: Explain the measured results. Separate derived values and architectural inference from direct observations.

## Connection to LLMs

TODO: Connect the verified result to inference behavior without claiming effects that were not measured.

## Further Exploration

TODO: Identify the next controlled experiment justified by the result.